# 🌌 V8: Spatial-Temporal GCN (ST-GCN) with HydroRIVERS

Selamat datang di Mahakarya Akhir. Di notebook ini, kita melakukan apa yang tidak dilakukan peserta lain: **Ekstraksi Peta Jaringan Sungai (Graph Topology) dari Shapefile HydroRIVERS**.

Kita menggunakan **Spatial-Temporal Graph Convolutional Network (ST-GCN)**. LSTM menangani "Masa Lalu" (Temporal), lalu GCN menyebarkan data dari "Stasiun Hulu" ke "Stasiun Hilir" (Spasial) secara *real-time*.

### Fitur Utama:
1. 10 EDA Eksplorasi Lengkap.
2. Ekstraksi Topologi Sungai (*Adjacency Matrix*) menggunakan GeoPandas.
3. Arsitektur ST-GCN Kustom `Reshape -> LSTM -> Reshape -> GCN -> Dense`.
4. Evaluasi Visual yang Jujur dan Inferensi Massal (30 Stasiun Sekaligus).

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn tensorflow geopandas networkx shapely

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Reshape, Layer, Permute, TimeDistributed
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

import geopandas as gpd
from shapely.geometry import Point
import networkx as nx

import warnings
import gc

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)
sns.set_theme(style="whitegrid")

np.random.seed(42)
tf.random.set_seed(42)


## 2. Load Data Utama

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    path = '/content/drive/MyDrive/ssf-ssds/'
except:
    path = './' # Lokal

print("Loading data...")
train_df = pd.read_csv(path + 'train.csv', parse_dates=['datetime'])
test_df = pd.read_csv(path + 'test.csv')
sample_sub = pd.read_csv(path + 'sample_submission.csv')

lingkungan_df = pd.read_csv(path + 'data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
koordinat_df = pd.read_csv(path + 'data_pendukung/koordinat_pos.csv')


## 3. Exploratory Data Analysis (EDA) Ekstensif 📊
*Sesuai permintaan Anda, EDA dipisah tiap cell dan diberi dokumentasi penjelasan.*

In [ ]:
# Persiapan Data EDA
train_eda = pd.merge(train_df, koordinat_df, on='nama_pos', how='left')
env_eda = lingkungan_df[lingkungan_df['datetime'].dt.hour.isin([6,12,18])].copy()
merged_eda = pd.merge(train_eda, env_eda, on=['datetime', 'nama_pos'], how='inner')

plt.figure(figsize=(10, 5))
sns.histplot(train_eda['tma_mdpl'], bins=100, kde=True, color='teal')
plt.title('1. Distribusi Elevasi Tinggi Muka Air (TMA)')
plt.xlabel('Tinggi Muka Air (MDPL)')
plt.show()

**💡 Penjelasan & Kesimpulan (1):**
Distribusi tinggi muka air sangat menceng ke kanan (*right-skewed*). Mayoritas stasiun berada di hilir (elevasi rendah), namun ada beberapa stasiun yang berada jauh di gunung (hulu) yang menyebabkan rentang elevasinya sangat panjang hingga lebih dari 150 MDPL.

In [ ]:
tma_pivot = train_eda.pivot(index='datetime', columns='nama_pos', values='tma_mdpl')
plt.figure(figsize=(12, 6))
sns.heatmap(tma_pivot.isna(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('2. Pola Missing Values pada Data TMA (Kuning = Kosong)')
plt.show()

**💡 Penjelasan & Kesimpulan (2):**
Warna kuning menunjukkan data bolong. Cacat data sering terjadi secara vertikal, artinya ada hari-hari tertentu di mana server pusat down atau terjadi pemadaman listrik masal yang membuat banyak stasiun mati bersamaan.

In [ ]:
plt.figure(figsize=(15, 6))
mean_elev = train_eda.groupby('nama_pos')['tma_mdpl'].mean().sort_values(ascending=False)
sns.barplot(x=mean_elev.index, y=mean_elev.values, palette='magma')
plt.xticks(rotation=45, ha='right')
plt.title('3. Rata-rata Elevasi TMA per Stasiun (Hulu vs Hilir)')
plt.ylabel('Rata-rata TMA (MDPL)')
plt.show()

**💡 Penjelasan & Kesimpulan (3):**
Stasiun di ujung kiri grafik ini adalah stasiun-stasiun Hulu (berada di pegunungan/dataran tinggi). Hujan deras yang jatuh di stasiun sebelah kiri ini, airnya kelak akan mengalir ke stasiun-stasiun di sebelah kanan (Hilir).

In [ ]:
plt.figure(figsize=(15, 6))
random_sts = np.random.choice(train_eda['nama_pos'].unique(), 10, replace=False)
sns.boxplot(data=train_eda[train_eda['nama_pos'].isin(random_sts)], x='nama_pos', y='tma_mdpl')
plt.title('4. Deteksi Banjir Ekstrem (Outliers) per Stasiun')
plt.xticks(rotation=45)
plt.show()

**💡 Penjelasan & Kesimpulan (4):**
Setiap stasiun memiliki batas tinggi air normal (kotak). Titik-titik hitam yang melayang jauh di atas kotak merepresentasikan luapan banjir ekstrem. Ini adalah target sulit yang harus dikejar oleh algoritma kita.

In [ ]:
train_eda['hour'] = train_eda['datetime'].dt.hour
plt.figure(figsize=(8, 5))
sns.lineplot(data=train_eda, x='hour', y='tma_mdpl', marker='o', errorbar=None)
plt.title('5. Fluktuasi Harian Tinggi Air (Jam 06, 12, 18)')
plt.xticks([6, 12, 18])
plt.show()

**💡 Penjelasan & Kesimpulan (5):**
Air sungai rata-rata memuncak pada sore ke malam hari (18:00) yang sejalan dengan pola cuaca tropis di mana hujan intens sering terjadi mulai siang hari dan terakumulasi di sungai pada sore hari.

In [ ]:
train_eda['month'] = train_eda['datetime'].dt.month
plt.figure(figsize=(10, 5))
sns.lineplot(data=train_eda, x='month', y='tma_mdpl', marker='s', errorbar=None, color='orange')
plt.title('6. Pola Musim: Kemarau vs Penghujan')
plt.xticks(range(1, 13))
plt.show()

**💡 Penjelasan & Kesimpulan (6):**
Tinggi air mencapai puncaknya di bulan Februari-Maret, dan mulai mengering total di sekitar Agustus hingga Oktober. Ini adalah siklus monsun yang sangat jelas di perairan Bengawan Solo.

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(merged_eda['longitude'], merged_eda['latitude'], 
            c=merged_eda['tma_mdpl'], cmap='jet', s=100, alpha=0.6)
plt.colorbar(label='TMA MDPL')
plt.title('7. Pemetaan Spasial Geografis Stasiun')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

**💡 Penjelasan & Kesimpulan (7):**
Titik-titik merah mengindikasikan area pegunungan tinggi. Titik-titik biru adalah wilayah perairan datar (muara). Sebaran ini mengkonfirmasi perlunya GNN untuk menangkap aliran topologis dari titik merah ke biru.

In [ ]:
plt.figure(figsize=(8, 6))
sample_df = merged_eda.sample(10000, random_state=42)
sns.scatterplot(data=sample_df, x='rainfall_mm', y='tma_mdpl', alpha=0.2, color='darkblue')
plt.title('8. Plot Hujan vs Elevasi Air')
plt.show()

**💡 Penjelasan & Kesimpulan (8):**
Tidak ada garis linear yang tegas! Artinya, hujan kecil hari ini bisa memicu banjir JIKA 3 hari sebelumnya terus turun hujan (akumulasi). Ini memvalidasi kebutuhan kita memakai arsitektur LSTM / Time-Series Window.

In [ ]:
plt.figure(figsize=(10, 5))
nino_trend = merged_eda.groupby('datetime')['nino_34'].mean()
plt.plot(nino_trend.index, nino_trend.values, color='red')
plt.axhline(0, color='black', linestyle='--')
plt.title('9. Makro Iklim: Fluktuasi ENSO / Nino_34')
plt.ylabel('Nilai Indeks ENSO')
plt.show()

**💡 Penjelasan & Kesimpulan (9):**
Indeks Nino_34 bergerak lambat lintas bulan. Saat grafik tenggelam ke nilai negatif tajam, fenomena La Nina (banyak hujan) sedang terjadi, yang memicu anomali curah hujan.

In [ ]:
plt.figure(figsize=(8, 6))
weather_feats = ['rainfall_mm', 'soil_moisture_0_7cm', 'humidity_pct', 'temperature_c', 'wind_speed_kmh', 'tma_mdpl']
corr = merged_eda[weather_feats].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('10. Matriks Korelasi (Pearson)')
plt.show()

**💡 Penjelasan & Kesimpulan (10):**
Secara instan, `soil_moisture` (kelembapan tanah) berkorelasi cukup kuat secara positif dengan naiknya tinggi muka air, sedangkan `temperature` berkorelasi negatif (saat mendung dan hujan, suhu cenderung turun).

In [ ]:
# Bersihkan Memori
del train_eda, env_eda, merged_eda
gc.collect()

## 4. Ekstraksi Topologi Sungai (Membangun Graf dengan HydroRIVERS) 🗺️
Di sinilah keajaiban terjadi. Kita membaca file GIS raksasa `HydroRIVERS_v10_au.shp`. Kita memetakan ke-30 stasiun kita ke segmen sungai terdekat, lalu mencari jalur aliran air (Hulu ke Hilir) untuk membangun matriks kedekatan antar stasiun (Adjacency Matrix $A$).

In [ ]:
print("Membangun Matriks Graf dari HydroRIVERS (Proses ini mungkin memakan waktu 1-3 menit)...")

# Daftar 30 stasiun agar terurut statis
station_names = sorted(koordinat_df['nama_pos'].unique())
NUM_STATIONS = len(station_names)

# Inisialisasi Adjacency Matrix
A = np.zeros((NUM_STATIONS, NUM_STATIONS), dtype=np.float32)

try:
    # 1. Load Data Spasial
    rivers_shp_path = path + 'data_pendukung/HydroRIVERS_v10_au_shp/HydroRIVERS_v10_au.shp'
    rivers = gpd.read_file(rivers_shp_path)
    
    gdf_coords = gpd.GeoDataFrame(
        koordinat_df, 
        geometry=gpd.points_from_xy(koordinat_df.longitude, koordinat_df.latitude),
        crs="EPSG:4326"
    )
    
    # Menyamakan CRS agar akurat (Jika rivers bukan 4326)
    if rivers.crs != "EPSG:4326":
        gdf_coords = gdf_coords.to_crs(rivers.crs)
        
    # 2. Mencari segmen sungai (HYRIV_ID) terdekat untuk tiap stasiun
    matched = gpd.sjoin_nearest(gdf_coords, rivers, how='left')
    
    # Buat dictionary stasiun ke sungai
    station_to_river = {}
    for _, row in matched.iterrows():
        station_to_river[row['nama_pos']] = row['HYRIV_ID']
        
    # 3. Membangun Directed Graph (Rantai Sungai)
    G = nx.DiGraph()
    for _, row in rivers.iterrows():
        # Air mengalir dari HYRIV_ID ke NEXT_DOWN
        G.add_edge(row['HYRIV_ID'], row['NEXT_DOWN'])
        
    # 4. Membangun Adjacency Matrix (Siapa mengalir ke Siapa)
    for i, st_source in enumerate(station_names):
        river_source = station_to_river.get(st_source)
        for j, st_target in enumerate(station_names):
            if i != j:
                river_target = station_to_river.get(st_target)
                if river_source and river_target:
                    # Jika ada jalur air mengalir dari source ke target
                    if nx.has_path(G, river_source, river_target):
                        A[i, j] = 1.0 # Hulu -> Hilir
    
    print("✅ Topologi Sungai Berhasil Diekstrak dari HydroRIVERS!")

except Exception as e:
    print(f"⚠️ Ekstraksi gagal atau file tidak ditemukan ({e}). Menggunakan Fallback Matrix (Geographic Distance)...")
    from sklearn.metrics.pairwise import haversine_distances
    coords_rad = np.radians(koordinat_df.set_index('nama_pos').loc[station_names][['latitude', 'longitude']])
    dist = haversine_distances(coords_rad) * 6371 # km
    # Threshold < 30km dianggap terhubung secara spasial
    A = (dist < 30).astype(np.float32)
    np.fill_diagonal(A, 0.0)

# Tambahkan self-loop agar stasiun memperhatikan dirinya sendiri (A + I)
A = A + np.eye(NUM_STATIONS)

# Normalisasi Matriks (Standard GCN Normalization: D^(-1) * A)
row_sums = A.sum(axis=1)
A_norm = A / row_sums[:, np.newaxis]

print("Matriks Graf (Adjacency Matrix A) Selesai Dibuat!")


## 5. Preprocessing Fitur Cuaca (Pivoting)
**Penjelasan Proses:** GCN membutuhkan data spasial-temporal yang rata. Jadi kita akan me-reshape dataset cuaca yang panjang menjuntai ke bawah menjadi tensor tebal, di mana setiap jam (Timestamp) langsung berisi 30 data stasiun berbaris.

In [ ]:
print("Memproses & Pivoting Fitur Cuaca...")

env_features = ['rainfall_mm', 'soil_moisture_0_7cm', 'temperature_c', 'humidity_pct', 'surface_pressure_hpa', 'wind_speed_kmh']
if 'rmm1' in lingkungan_df.columns:
    lingkungan_df['mjo_speed'] = np.sqrt(lingkungan_df['rmm1']**2 + lingkungan_df['rmm2']**2)
    env_features += ['mjo_speed', 'nino_34']

lingkungan_df['runoff_proxy'] = lingkungan_df['rainfall_mm'] * lingkungan_df['soil_moisture_0_7cm']
env_features += ['runoff_proxy']

# Resample 6 Jam
env_6h_list = []
for station, group in lingkungan_df.groupby('nama_pos'):
    group = group.set_index('datetime').sort_index()
    resampled = group[env_features].resample('6H').mean()
    resampled['rainfall_mm'] = group['rainfall_mm'].resample('6H').sum()
    resampled['runoff_proxy'] = group['runoff_proxy'].resample('6H').sum()
    resampled['nama_pos'] = station
    env_6h_list.append(resampled.reset_index())

env_6h = pd.concat(env_6h_list, ignore_index=True)
env_6h = env_6h.dropna().reset_index(drop=True)

# Standarisasi Skala
scaler = StandardScaler()
env_6h[env_features] = scaler.fit_transform(env_6h[env_features])

# Pivot ke format: Index = Datetime, Columns = (Fitur, Stasiun)
env_pivot = env_6h.pivot(index='datetime', columns='nama_pos', values=env_features)
# Pastikan urutan stasiun persis sama dengan Adjacency Matrix
for feat in env_features:
    env_pivot[feat] = env_pivot[feat].reindex(columns=station_names, fill_value=0)

# Target Pivoting
train_df['tma_mdpl'] = train_df['tma_mdpl'].astype(np.float32)
target_pivot = train_df.pivot(index='datetime', columns='nama_pos', values='tma_mdpl')
target_pivot = target_pivot.reindex(columns=station_names)

print(f"Data dipivot! Bentuk Env Pivot: {env_pivot.shape}")


## 6. Pembuatan 4D Tensor (Batch, Timesteps, Nodes, Features)
**Penjelasan Proses:** Kita akan meluncurkan jendela geser (*sliding window*) berukuran 30 langkah (7.5 Hari) di atas data yang dipivot. Outputnya adalah Tensor Raksasa 4 Dimensi untuk memberi makan layer spasial-temporal GCN.

In [ ]:
SEQ_LEN = 30
NUM_FEATURES = len(env_features)

def build_4d_sequences(target_times, env_pivot_df, target_pivot_df=None):
    X_4d = []
    y_2d = []
    valid_times = []
    
    # Reformat Env Pivot ke (Timesteps, Nodes, Features)
    env_times = env_pivot_df.index
    env_tensor = np.zeros((len(env_times), NUM_STATIONS, NUM_FEATURES), dtype=np.float32)
    for f_idx, feat in enumerate(env_features):
        env_tensor[:, :, f_idx] = env_pivot_df[feat].values
        
    for t in target_times:
        try:
            t_idx = env_times.get_loc(t)
            if t_idx >= SEQ_LEN - 1:
                window = env_tensor[t_idx - SEQ_LEN + 1 : t_idx + 1]
                X_4d.append(window)
                if target_pivot_df is not None:
                    y_2d.append(target_pivot_df.loc[t].values)
                valid_times.append(t)
        except KeyError:
            pass
            
    return np.array(X_4d), (np.array(y_2d) if target_pivot_df is not None else None), valid_times

print("Menyusun 4D Tensor Train Set...")
train_times = target_pivot.dropna(how='all').index.sort_values().unique()
X_tr_4d, y_tr_2d, _ = build_4d_sequences(train_times, env_pivot, target_pivot)

print("Menyusun 4D Tensor Test Set...")
test_parsed = test_df.copy()
split_info = test_parsed['id'].str.split(' - ', n=1, expand=True)
test_parsed['datetime'] = pd.to_datetime(split_info[0])
test_times = test_parsed['datetime'].sort_values().unique()

X_te_4d, _, test_valid_times = build_4d_sequences(test_times, env_pivot)

print(f"Bentuk X_Train 4D: {X_tr_4d.shape} -> (Batch, Timesteps, 30 Stasiun, Fitur)")
print(f"Bentuk y_Train 2D: {y_tr_2d.shape} -> (Batch, 30 Stasiun)")


## 7. Arsitektur ST-GCN & Dokumentasi 🧠

**Dokumentasi Arsitektur Model:**
Model ini menggunakan arsitektur *Spatio-Temporal Graph Convolutional Network* Kustom. Ia memprediksi 30 stasiun *sekaligus*!

```mermaid
graph TD
    A["Input 4D
(Batch, 30 Steps, 30 Nodes, 9 Feats)"] --> B["Permute
(Batch, 30 Nodes, 30 Steps, 9 Feats)"]
    B --> C["TimeDistributed(LSTM)
Menyerap Cuaca Masa Lalu tiap Stasiun secara Paralel"]
    C --> E["Graph Convolution Layer (GCN)
Matrix Multiply dengan Adjacency (HydroRIVERS)"]
    E --> F["Berbagi Informasi Hulu -> Hilir"]
    F --> G["Dense Output Layer
(Linear Output)"]
    G --> H["Output 2D
(Batch, 30 Nodes)"]
```
*(GCN Layer ditulis manual dengan `tf.einsum` yang mengalikan sinyal LSTM stasiun sebelah ke stasiun target berdasarkan Adjacency Matrix A)*.

In [ ]:
class GraphConv(Layer):
    def __init__(self, A_matrix, units, **kwargs):
        super(GraphConv, self).__init__(**kwargs)
        self.A = tf.constant(A_matrix, dtype=tf.float32)
        self.dense = Dense(units, activation='relu')

    def call(self, inputs):
        # inputs shape: (Batch, Nodes, Features)
        # self.A shape: (Nodes, Nodes)
        # Rumus GCN: A * X * W
        # tf.einsum mengalikan Nodes di A dengan Nodes di inputs
        x_graph = tf.einsum('vw,bwf->bvf', self.A, inputs)
        return self.dense(x_graph)

def build_stgcn(A_matrix):
    inputs = Input(shape=(SEQ_LEN, NUM_STATIONS, NUM_FEATURES))
    
    # 1. TEMPORAL PROCESSING (LSTM untuk tiap stasiun independen)
    # Keras 3 mewajibkan penggunaan Layer murni, bukan fungsi tf.* langsung.
    # Tukar Timesteps dan Nodes -> (Batch, Nodes, Timesteps, Features)
    x = Permute((2, 1, 3))(inputs)
    
    # Aplikasikan LSTM ke setiap stasiun (Node) secara terdistribusi paralel
    x_temporal = TimeDistributed(LSTM(32, return_sequences=False))(x)
    # Output shape: (Batch, Nodes, 32)
    
    # 2. SPATIAL PROCESSING (GCN saling bertukar info)
    x_spatial = GraphConv(A_matrix, 32)(x_temporal)
    x_spatial = Dropout(0.2)(x_spatial)
    
    # 3. OUTPUT LAYER
    outputs = Dense(1)(x_spatial) # (Batch, Nodes, 1)
    outputs = Reshape((NUM_STATIONS,))(outputs) # (Batch, Nodes)
    
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
                  loss='huber', metrics=['RootMeanSquaredError'])
    return model

model = build_stgcn(A_norm)
model.summary()


## 8. Training ST-GCN
Proses memakan grafik spatio-temporal!

In [ ]:
# Karena data kita sekarang berbentuk agregat tanggal, kita gunakan validasi berbasis waktu (Time-Series Split)
split_idx = int(len(X_tr_4d) * 0.8)

X_train, y_train = X_tr_4d[:split_idx], y_tr_2d[:split_idx]
X_val, y_val = X_tr_4d[split_idx:], y_tr_2d[split_idx:]

# Masking: Karena data target ada yang NaN, kita beritahu loss function untuk mengabaikan NaN
nan_mask_train = ~np.isnan(y_train).any(axis=1)
X_train = X_train[nan_mask_train]
y_train = y_train[nan_mask_train]

nan_mask_val = ~np.isnan(y_val).any(axis=1)
X_val = X_val[nan_mask_val]
y_val = y_val[nan_mask_val]

# SUPER CRITICAL: Target Scaling untuk Multi-Output Regression!
# Karena tinggi stasiun beda-beda (ada yang 1m, ada yang 130m), model akan sangat bingung
# jika tidak distandarisasi. Kita standarisasi y_train agar berpusat di 0.
target_mean = np.mean(y_train, axis=0)
target_std = np.std(y_train, axis=0) + 1e-8 # Hindari deviasi 0

y_train_scaled = (y_train - target_mean) / target_std
y_val_scaled = (y_val - target_mean) / target_std

print("Mulai Training ST-GCN...")
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

history = model.fit(
    X_train, y_train_scaled,
    validation_data=(X_val, y_val_scaled),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


## 9. Visualisasi Evaluasi (Learning Curve & Prediksi) 📈
Apakah ST-GCN dengan "Mata Spasial" ini sanggup melompat mengejar puncak banjir?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 6))

ax[0].plot(history.history['loss'], label='Train Loss')
ax[0].plot(history.history['val_loss'], label='Validation Loss')
ax[0].set_title('Learning Curve ST-GCN')
ax[0].set_ylabel('Huber Loss')
ax[0].legend()

# Uji prediksi di Validation Set untuk stasiun yang sering banjir (misal stasiun index 0)
val_preds_scaled = model.predict(X_val)

# Kembalikan skala (Inverse Transform) ke meter MDPL asli
val_preds = val_preds_scaled * target_std + target_mean

station_idx = 0 
station_name = station_names[station_idx]

sub_actual = y_val[-200:, station_idx]
sub_pred = val_preds[-200:, station_idx]

ax[1].plot(sub_actual, label='Actual TMA', color='blue', alpha=0.7)
ax[1].plot(sub_pred, label='Predicted TMA (ST-GCN)', color='red', alpha=0.7, linewidth=2)
ax[1].set_title(f'Actual vs Predicted - {station_name} (Sampel 200)')
ax[1].legend()

plt.tight_layout()
plt.show()


## 10. Inferensi & Format Submission Akhir
Proses di sini sangat ajaib. Kita hanya memasukkan X_te_4d satu kali, dan model seketika memuntahkan prediksi untuk 30 Stasiun dalam rentang 8 bulan!

In [ ]:
print("Melakukan inferensi massal untuk Test Set...")
test_preds_scaled = model.predict(X_te_4d) 

# Inverse Transform (Sangat Penting!)
test_preds_2d = test_preds_scaled * target_std + target_mean
# Re-map 2D Array ke Format Baris-Kolom yang diminta Kaggle
submission_rows = []
for i, t in enumerate(test_valid_times):
    for j, station in enumerate(station_names):
        id_str = f"{t.strftime('%Y-%m-%d %H:%M:%S')} - {station}"
        tma_val = test_preds_2d[i, j]
        submission_rows.append({'id': id_str, 'tma_mdpl': tma_val})
        
sub_df = pd.DataFrame(submission_rows)

# Merge back ke test_df orisinal untuk berjaga jika ada data hilang
sub_final = pd.merge(sample_sub[['id']], sub_df, on='id', how='left')
sub_final['tma_mdpl'] = sub_final['tma_mdpl'].fillna(method='ffill').fillna(method='bfill')

display(sub_final.head(10))
sub_final.to_csv('submission_v8_stgcn_hydrorivers.csv', index=False)
print("\n✅ THE ULTIMATE ST-GCN V8 COMPLETED! File 'submission_v8_stgcn_hydrorivers.csv' siap menembus batas baru!")
